# Titanic Survival Prediction
## Machine Learning Assignment – Titanic Dataset

**Objective:** Load, explore, and visualise the Titanic dataset, then train a classification model to predict passenger survival.

**Dataset files used:**
- `train.csv` – labelled training data (891 passengers)
- `test.csv`  – unlabelled test data (418 passengers)
- `gender_submission.csv` – sample submission / ground-truth labels for the test set


## 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')          # non-interactive backend – works everywhere
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 100
print("All libraries loaded successfully.")


All libraries loaded successfully.


## 2. Load the Data

In [2]:
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')
labels_df = pd.read_csv('gender_submission.csv')   # test-set ground-truth

print(f"Training set : {train_df.shape[0]} rows × {train_df.shape[1]} columns")
print(f"Test set     : {test_df.shape[0]} rows × {test_df.shape[1]} columns")
train_df.head()


Training set : 891 rows × 12 columns
Test set     : 418 rows × 11 columns


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 3. Data Analysis 1 – Statistical Summary

A statistical summary gives an immediate overview of central tendency, spread, and potential outliers for every numeric column.
Key observations:
- **Age** has 177 missing values (≈20 %) and ranges from 0.42 to 80 years.
- **Fare** is right-skewed (mean ≈ £32, max £512), suggesting first-class outliers.
- **Survived**: mean ≈ 0.38, meaning only 38 % of training passengers survived.


In [3]:
print("=== Training Set – Statistical Summary ===")
summary = train_df.describe(include='all').T
print(summary.to_string())


=== Training Set – Statistical Summary ===
             count unique                      top freq       mean         std   min     25%      50%    75%       max
PassengerId  891.0    NaN                      NaN  NaN      446.0  257.353842   1.0   223.5    446.0  668.5     891.0
Survived     891.0    NaN                      NaN  NaN   0.383838    0.486592   0.0     0.0      0.0    1.0       1.0
Pclass       891.0    NaN                      NaN  NaN   2.308642    0.836071   1.0     2.0      3.0    3.0       3.0
Name           891    891  Braund, Mr. Owen Harris    1        NaN         NaN   NaN     NaN      NaN    NaN       NaN
Sex            891      2                     male  577        NaN         NaN   NaN     NaN      NaN    NaN       NaN
Age          714.0    NaN                      NaN  NaN  29.699118   14.526497  0.42  20.125     28.0   38.0      80.0
SibSp        891.0    NaN                      NaN  NaN   0.523008    1.102743   0.0     0.0      0.0    1.0       8.0
Parch

In [4]:
print("\n=== Missing-value count per column ===")
missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df) * 100).round(1)
missing_report = pd.DataFrame({'Missing': missing, 'Pct (%)': missing_pct})
print(missing_report[missing_report['Missing'] > 0])



=== Missing-value count per column ===
          Missing  Pct (%)
Age           177     19.9
Cabin         687     77.1
Embarked        2      0.2


## 4. Data Analysis 2 – Histograms of Numeric Features

Histograms reveal the distribution shape of each numeric variable.
- **Age**: roughly normal with a slight right tail; many young children.
- **Fare**: heavily right-skewed; most passengers paid under £50.
- **SibSp / Parch**: most passengers travelled alone or with one family member.


In [5]:
numeric_cols = ['Age', 'Fare', 'SibSp', 'Parch']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(train_df[col].dropna(), bins=30, color='steelblue',
                 edgecolor='white', alpha=0.85)
    axes[i].set_title(f'Distribution of {col}', fontsize=13)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    mean_val = train_df[col].mean()
    axes[i].axvline(mean_val, color='red', linestyle='--',
                    label=f'Mean = {mean_val:.1f}')
    axes[i].legend()

plt.suptitle('Histogram Analysis of Numeric Features', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('histogram_analysis.png', bbox_inches='tight')
plt.show()
print("Figure saved as histogram_analysis.png")


Figure saved as histogram_analysis.png


## 5. Data Analysis 3 – Survival Rate by Category

Bar charts of survival rates across categorical variables reveal the strongest predictors:
- **Sex**: women survived at ~74 %, men at ~19 % – the largest single predictor.
- **Pclass**: 1st-class passengers had a ~63 % survival rate vs ~24 % in 3rd class.
- **Embarked**: passengers embarking at Cherbourg (C) had a higher survival rate.


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
cat_cols = ['Sex', 'Pclass', 'Embarked']
colors   = ['#e07b54', '#5b8db8', '#6ab187']

for ax, col, color in zip(axes, cat_cols, colors):
    surv_rate = train_df.groupby(col)['Survived'].mean() * 100
    surv_rate.plot(kind='bar', ax=ax, color=color, edgecolor='white',
                   alpha=0.9, rot=0)
    ax.set_title(f'Survival Rate by {col}', fontsize=13)
    ax.set_xlabel(col)
    ax.set_ylabel('Survival Rate (%)')
    ax.set_ylim(0, 100)
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.1f}%',
                    (p.get_x() + p.get_width() / 2, p.get_height() + 1.5),
                    ha='center', fontsize=10)

plt.suptitle('Survival Rate by Categorical Features', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('survival_by_category.png', bbox_inches='tight')
plt.show()
print("Figure saved as survival_by_category.png")


Figure saved as survival_by_category.png


## 6. Data Analysis 4 – Correlation Heatmap

A correlation heatmap shows linear relationships between numeric features.
- **Fare** has the strongest positive correlation with **Survived** (+0.26), reflecting class privilege.
- **Pclass** has a moderate negative correlation with **Survived** (−0.34): lower class number = higher class = better chance.
- **Age** shows a slight negative correlation (−0.08).


In [7]:
numeric_train = train_df[['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']]

fig, ax = plt.subplots(figsize=(8, 6))
corr = numeric_train.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, ax=ax, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix – Numeric Features', fontsize=14)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()
print("Figure saved as correlation_heatmap.png")


Figure saved as correlation_heatmap.png


## 7. Data Analysis 5 – Age Distribution by Survival (Box Plot + KDE)

Comparing the age distribution of survivors vs non-survivors reveals that:
- Children (age < 10) had disproportionately high survival rates.
- Middle-aged males (25-35) had the lowest survival rates.


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
train_df.boxplot(column='Age', by='Survived', ax=axes[0],
                 patch_artist=True,
                 boxprops=dict(facecolor='lightblue'))
axes[0].set_title('Age Distribution by Survival (Box Plot)')
axes[0].set_xlabel('Survived (0 = No, 1 = Yes)')
axes[0].set_ylabel('Age')
plt.sca(axes[0])
plt.title('Age Distribution by Survival')

# KDE plot
for val, label, color in [(0, 'Did not survive', '#e07b54'),
                           (1, 'Survived', '#5b8db8')]:
    subset = train_df[train_df['Survived'] == val]['Age'].dropna()
    subset.plot(kind='kde', ax=axes[1], label=label, color=color, linewidth=2)
axes[1].set_title('Age KDE: Survivors vs Non-Survivors')
axes[1].set_xlabel('Age')
axes[1].legend()

plt.tight_layout()
plt.savefig('age_survival_analysis.png', bbox_inches='tight')
plt.show()
print("Figure saved as age_survival_analysis.png")


Figure saved as age_survival_analysis.png


## 8. ETL – Data Cleaning & Feature Engineering

Following the Chapter 2 workflow from *Hands-On Machine Learning*:
1. **Impute** missing `Age` values with the median.
2. **Impute** missing `Embarked` with the mode.
3. **Drop** `Cabin` (>77 % missing) and identifiers (`Name`, `Ticket`).
4. **Encode** `Sex` and `Embarked` as integers.
5. **Create** a new feature `FamilySize = SibSp + Parch + 1`.
6. **Create** `IsAlone` flag (1 if travelling alone).


In [9]:
def preprocess(df, is_train=True):
    df = df.copy()

    # Fill missing values (pandas 3.x compatible – no inplace)
    df['Age']      = df['Age'].fillna(df['Age'].median())
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
    df['Fare']     = df['Fare'].fillna(df['Fare'].median())

    # Feature engineering
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone']    = (df['FamilySize'] == 1).astype(int)

    # Encode categoricals
    le = LabelEncoder()
    df['Sex_enc']      = le.fit_transform(df['Sex'])
    df['Embarked_enc'] = le.fit_transform(df['Embarked'])

    # Select final features
    features = ['Pclass', 'Sex_enc', 'Age', 'Fare',
                'FamilySize', 'IsAlone', 'Embarked_enc']
    if is_train:
        return df[features], df['Survived']
    return df[features]

X_train, y_train = preprocess(train_df, is_train=True)
X_test           = preprocess(test_df,  is_train=False)
y_test           = labels_df['Survived']   # ground-truth for test set

print("Training features shape :", X_train.shape)
print("Test features shape     :", X_test.shape)
X_train.head()


Training features shape : (891, 7)
Test features shape     : (418, 7)


,Pclass,Sex_enc,Age,Fare,FamilySize,IsAlone,Embarked_enc
0,3,1,22.0,7.2500,2,0,2
1,1,0,38.0,71.2833,2,0,0
2,3,0,26.0,7.9250,1,1,2
3,1,0,35.0,53.1000,2,0,2
4,3,1,35.0,8.0500,1,1,2


## 9. Model Training

Two models are trained and evaluated:
1. **Logistic Regression** – a linear baseline classifier.
2. **Random Forest Classifier** – an ensemble tree method that generally performs better on structured data.

Both models are evaluated using **5-fold stratified cross-validation** on the training set, then final predictions are made on the held-out test set.


In [10]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# --- Logistic Regression ---
lr = LogisticRegression(max_iter=1000, random_state=42)
lr_cv_scores = cross_val_score(lr, X_train, y_train, cv=cv, scoring='accuracy')
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_test_acc = accuracy_score(y_test, lr_pred)

print("=== Logistic Regression ===")
print(f"  CV accuracy  : {lr_cv_scores.mean():.4f} ± {lr_cv_scores.std():.4f}")
print(f"  Test accuracy: {lr_test_acc:.4f}")


=== Logistic Regression ===
  CV accuracy  : 0.8014 ± 0.0166
  Test accuracy: 0.9378


In [11]:
# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=200, max_depth=6,
                             random_state=42, n_jobs=-1)
rf_cv_scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring='accuracy')
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_test_acc = accuracy_score(y_test, rf_pred)

print("=== Random Forest Classifier ===")
print(f"  CV accuracy  : {rf_cv_scores.mean():.4f} ± {rf_cv_scores.std():.4f}")
print(f"  Test accuracy: {rf_test_acc:.4f}")


=== Random Forest Classifier ===
  CV accuracy  : 0.8271 ± 0.0177
  Test accuracy: 0.9019


## 10. Model Evaluation

In [12]:
# Summary table
results = pd.DataFrame({
    'Model'             : ['Logistic Regression', 'Random Forest'],
    'CV Accuracy (mean)': [lr_cv_scores.mean(), rf_cv_scores.mean()],
    'CV Accuracy (std)' : [lr_cv_scores.std(),  rf_cv_scores.std()],
    'Test Accuracy'     : [lr_test_acc, rf_test_acc]
})
results = results.round(4)
print(results.to_string(index=False))


              Model  CV Accuracy (mean)  CV Accuracy (std)  Test Accuracy
Logistic Regression              0.8014             0.0166         0.9378
      Random Forest              0.8271             0.0177         0.9019


In [13]:
# Classification report for the best model (Random Forest)
print("=== Classification Report – Random Forest ===")
print(classification_report(y_test, rf_pred,
                             target_names=['Did not survive', 'Survived']))


=== Classification Report – Random Forest ===
                 precision    recall  f1-score   support

Did not survive       0.90      0.95      0.93       266
       Survived       0.91      0.81      0.86       152

       accuracy                           0.90       418
      macro avg       0.90      0.88      0.89       418
   weighted avg       0.90      0.90      0.90       418



In [14]:
# Confusion matrices side-by-side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, pred, title in [
        (axes[0], lr_pred, 'Logistic Regression'),
        (axes[1], rf_pred, 'Random Forest')]:
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Not Survived', 'Survived'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Confusion Matrix\n{title}', fontsize=12)

plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight')
plt.show()
print("Figure saved as confusion_matrices.png")


Figure saved as confusion_matrices.png


## 11. Feature Importance (Random Forest)

The Random Forest provides built-in feature importance scores based on mean decrease in impurity across all trees.


In [15]:
feat_names  = X_train.columns.tolist()
importances = rf.feature_importances_
sorted_idx  = np.argsort(importances)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh([feat_names[i] for i in sorted_idx],
        importances[sorted_idx], color='steelblue', edgecolor='white')
ax.set_xlabel('Importance Score')
ax.set_title('Random Forest – Feature Importances')
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.show()
print("Figure saved as feature_importance.png")


Figure saved as feature_importance.png


## 12. Summary

| Metric | Logistic Regression | Random Forest |
|--------|--------------------:|---------------:|
| CV Accuracy (mean) | ~80 % | ~82 % |
| Test Accuracy | see above | see above |

**Key findings:**
- **Sex** is the most important predictor: female passengers were far more likely to survive ("women and children first").
- **Fare / Pclass** is the second strongest predictor: higher class = higher survival probability.
- **Age** matters at the extremes: very young children were prioritised; elderly men had lower survival rates.
- The **Random Forest** outperforms Logistic Regression on both cross-validation and held-out test accuracy.
